# HW1. Решение: инференс всех моделей

Ноутбук загружает обученные модели из `artifacts/` и сохраняет их ответы на диалогах из `HW1_DATA`.
Функции лежат рядом, в `solution_utils.py`, ноутбук их только импортирует и вызывает. Модели обучает и сравнивает
ноутбук `kirp_task1_improved.ipynb`.

**Что внутри сейчас:** задача 1 (реакция пользователя). Цикл по моделям задачи 2 (темы) добавляется отдельной секцией
между задачей 1 и сохранением ответов (место отмечено ниже).

Метрики ниже считаются на `HW1_DATA`; если это обучающий файл, они завышены (модели на нем обучались) и нужны только
как проверка, что все работает. Честные цифры на новых данных — в ноутбуке обучения (тестовый сет) и в отчете.
Пути заданы в первой ячейке кода, при проверке подставляются через переменные окружения.

In [ ]:
import os

import pandas as pd

from solution_utils import (
    compute_metrics, init_reaction_models, predict_reaction, read_dialogs, read_labels, save_predictions,
)

In [ ]:
TRAIN_PATH = os.environ.get("HW1_TRAIN", "hw1_train.jsonl")
DATA_PATH = os.environ.get("HW1_DATA", "hw1_train.jsonl")
LABELS_PATH = os.environ.get("HW1_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW1_PREDICTIONS", "predictions.csv")

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))

## Задача 1. Реакция пользователя

* `most_frequent`, `stratified` — заглушки, нижняя граница качества;
* `bow_logreg`, `tfidf_logreg`, `tfidf_cnb` — базовые: мешок слов или TF-IDF по тексту диалога и классификатор;
* `kirp_model` — TF-IDF и логистическая регрессия с подобранной предобработкой;
* `turn_level_logreg` — классификатор реплик пользователя по тексту реплики и ответу модели; ответ по диалогу —
  реакция последней не нейтральной реплики.

In [ ]:
reaction_models = init_reaction_models(TRAIN_PATH)
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

reaction_predictions, reaction_metrics = {}, {}
for name, model in reaction_models.items():
    reaction_predictions[name] = predict_reaction(model, dialogs)
    if reaction_labels:
        reaction_metrics[name] = compute_metrics(dialogs, reaction_predictions[name], reaction_labels)

pd.DataFrame(reaction_metrics).T.round(3) if reaction_metrics else "метки не переданы, метрики не считаются"

## Задача 2. Темы

Эта секция пока пустая: сюда добавляется цикл по моделям задачи 2 (`init_topic_models`, `predict_topic`).
До тех пор ответы по темам не сохраняются.

In [ ]:
topic_predictions = {}  # задача 2: имя модели -> список ответов (пока пусто)

## Ответы всех моделей

Последняя ячейка сохраняет ответы всех моделей в csv: `dialog_id`, `task`, `model`, `prediction`.

In [ ]:
save_predictions(PREDICTIONS_PATH, dialogs, {"reaction": reaction_predictions, "topic": topic_predictions})
n_models = len(reaction_predictions) + len(topic_predictions)
print("строк в файле ответов:", len(dialogs) * n_models)